# OpenAI API con Python

**Conceptos:** hacer llamadas a la API, usar instrucciones, JSON estructurado, y tools

## 1. Setup

Instalar dependencias.

Ejecutar una vez.

In [ ]:
%pip install --upgrade openai pydantic

## 2. API Key

Usar variable de entorno.

No hardcodear secrets.

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")

from openai import OpenAI

client = OpenAI()

print(f"Modelo: {OPENAI_MODEL}")
print("Cliente listo")

## 3. Primera llamada

Caso mínimo: enviar texto y leer respuesta.

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input="Explica qué es una API en una frase simple."
)

print(response.output_text)

## 4. Instrucciones

`instructions` define rol, tono y formato.

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    instructions="Sos un profesor de programación. Explicá claro y breve.",
    input="Explica qué es una API REST con un ejemplo."
)

print(response.output_text)

## 5. Mensajes con roles

Útil para separar reglas de la app y pedido del usuario.

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input=[
        {
            "role": "developer",
            "content": "Sos un asistente técnico. Respondé en español, breve y accionable."
        },
        {
            "role": "user",
            "content": "Tengo error 401 al llamar una API. ¿Qué reviso?"
        }
    ]
)

print(response.output_text)

## 6. JSON estructurado

Usar Pydantic para recibir datos parseables.

In [ ]:
from typing import Literal
from pydantic import BaseModel


class TicketClasificado(BaseModel):
    categoria: Literal["login", "pagos", "bug", "consulta", "otro"]
    prioridad: Literal["baja", "media", "alta"]
    resumen: str
    respuesta_cliente: str


ticket = """
No puedo entrar a mi cuenta desde ayer.
Probé cambiar la contraseña, pero nunca me llega el email.
Necesito acceder urgente porque tengo una factura pendiente.
"""

response = client.responses.parse(
    model=OPENAI_MODEL,
    input=[
        {
            "role": "developer",
            "content": "Clasifica tickets de soporte y genera una respuesta breve."
        },
        {
            "role": "user",
            "content": ticket
        }
    ],
    text_format=TicketClasificado,
)

resultado = response.output_parsed

print(resultado.model_dump_json(indent=2))

## 7. Ejercicio: extractor de factura

Completar el esquema y probar con texto libre.

In [ ]:
from pydantic import BaseModel


class FacturaExtraida(BaseModel):
    proveedor: str
    monto: float
    moneda: str
    fecha: str
    items: list[str]


texto_factura = """
Factura de Acme SA.
Fecha: 2026-05-10.
Total: USD 1250.50.
Items: hosting anual, soporte premium, dominio.
"""

response = client.responses.parse(
    model=OPENAI_MODEL,
    instructions="Extrae datos de facturas. No inventes campos.",
    input=texto_factura,
    text_format=FacturaExtraida,
)

factura = response.output_parsed

print(factura.model_dump_json(indent=2))

## 8. Function calling

El modelo pide una función.

Python ejecuta la función.

In [ ]:
import json


def buscar_pedido(id_pedido: str):
    pedidos = {
        "123": {
            "id": "123",
            "estado": "en camino",
            "fecha_estimada": "2026-05-18",
            "empresa_envio": "Correo Demo"
        },
        "456": {
            "id": "456",
            "estado": "pendiente de pago",
            "fecha_estimada": None,
            "empresa_envio": None
        }
    }

    return pedidos.get(
        id_pedido,
        {
            "id": id_pedido,
            "estado": "no encontrado"
        }
    )


tools = [
    {
        "type": "function",
        "name": "buscar_pedido",
        "description": "Busca el estado de un pedido por ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "id_pedido": {
                    "type": "string",
                    "description": "ID del pedido. Ejemplo: 123."
                }
            },
            "required": ["id_pedido"],
            "additionalProperties": False
        },
        "strict": True
    }
]


input_messages = [
    {
        "role": "user",
        "content": "Hola, quiero saber el estado de mi pedido 123."
    }
]

response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_messages,
    tools=tools
)

input_messages += response.output

for item in response.output:
    if item.type == "function_call" and item.name == "buscar_pedido":
        args = json.loads(item.arguments)
        resultado = buscar_pedido(id_pedido=args["id_pedido"])

        input_messages.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(resultado, ensure_ascii=False)
        })

final_response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_messages,
    tools=tools,
    instructions="Responde al cliente de forma clara y amable."
)

print(final_response.output_text)

## 9. Web search

Usar cuando la respuesta necesita información actual.

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "web_search"}],
    input="Busca una noticia tecnológica positiva de hoy y resúmela en 3 bullets."
)

print(response.output_text)